# Historical Wildfire Analysis on or Near Tribal Lands

**Series:** Tribal Fire Science & Indigenous Data Sovereignty  
**Author:** Lilly Jones, PhD  
**Last Updated:** 2025  
**Data Sources:** MTBS, Census TIGER AIANNH

## Overview

This notebook analyzes historical wildfire data (1984–present) from the Monitoring Trends
in Burn Severity (MTBS) dataset, focusing on fires occurring on or near Tribal lands
as defined by the US Census Bureau TIGER/Line AIANNH boundaries. It examines temporal
trends, per-tribe fire frequency and severity, and a composite fire risk score.

## Research Questions
- Which tribal lands have experienced the highest fire frequency and burned acreage since 1984?
- How have fire frequency and total acres burned on tribal lands changed over time?
- Which fires represent the greatest risk by combining frequency and average fire size?

## Data Sovereignty Note
> Tribal boundary data used here originates from Federal census records and may not
> reflect Tribal Nations' own definitions of territory. See the acknowledgment below.
> This notebook uses real data from documented public sources only.

In [ ]:
# Set Environment
import sys
from pathlib import Path

# Resolve repo root (one level above /notebooks)
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Imports
import warnings
import zipfile
from io import BytesIO

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

# Project modules
from src.data import constants, loaders, validators
from src.geo import utils as geo_utils
from src.indigenous.sovereignty import (
    generate_citations,
    print_data_acknowledgment,
)
from src.viz import charts, styles

styles.apply_mpl_style()
%matplotlib inline

# Suppress only known noisy warnings; leave errors visible
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {constants.DATA_DIR}")
print(f"Output dir: {constants.OUTPUTS_DIR}")

In [ ]:
# Data sovereignty acknowledgment
print_data_acknowledgment(source_keys=["census_aiannh", "mtbs"])

## Load Data

In [ ]:
# Tribal lands from Census TIGER AIANNH
# Downloads and caches automatically on first run.
tribal_lands = loaders.load_census_aian()
tribal_lands = validators.validate_geodataframe(
    tribal_lands,
    dataset_name="census_aiannh",
    required_columns=["geometry", "NAME", "NAMELSAD"],
)
print(f"Tribal land areas loaded: {len(tribal_lands):,}")
tribal_lands[["NAME", "NAMELSAD", "AIANNHCE"]].head()

In [ ]:
# MTBS fire perimeters
# Place the MTBS shapefile in data/raw/mtbs_perimeters/ and name it
# mtbs_perims_DD.shp — OR let the loader download it automatically.
#
# If you already have the file locally, copy it:
#   cp /your/old/path/mtbs_perims_DD.shp data/raw/mtbs_perimeters/
#
# The loader checks data/cache/ first, then data/raw/, then downloads.

MTBS_LOCAL = constants.RAW_DIR / "mtbs_perimeters" / "mtbs_perims_DD.shp"

if MTBS_LOCAL.exists():
    print(f"Loading MTBS from local file: {MTBS_LOCAL.relative_to(REPO_ROOT)}")
    fire_gdf = gpd.read_file(MTBS_LOCAL)
else:
    print("Local MTBS file not found — downloading via loader (large file, may take a few minutes)...")
    fire_gdf = loaders.load_mtbs_perimeters()

# Normalize column names to lowercase
fire_gdf.columns = fire_gdf.columns.str.lower()

# Parse ignition date and derive year
fire_gdf["ig_date"] = pd.to_datetime(fire_gdf["ig_date"], errors="coerce")
fire_gdf["fire_year"] = fire_gdf["ig_date"].dt.year

# Standardize acreage column name
if "burnbndac" in fire_gdf.columns:
    fire_gdf["fire_size_acres"] = fire_gdf["burnbndac"]

# Size classification (NWCG standard)
fire_gdf["size_class"] = pd.cut(
    fire_gdf["fire_size_acres"],
    bins=[0, 10, 100, 300, 1000, 5000, float("inf")],
    labels=["A", "B", "C", "D", "E", "F"],
)

fire_gdf = validators.validate_geodataframe(
    fire_gdf,
    dataset_name="mtbs_perimeters",
    required_columns=["geometry", "ig_date", "fire_size_acres"],
)

print(f"MTBS fire perimeters loaded: {len(fire_gdf):,}")
print(f"Year range: {fire_gdf['fire_year'].min()} – {fire_gdf['fire_year'].max()}")
print(f"CRS: {fire_gdf.crs}")
fire_gdf[["event_id", "incid_name", "fire_year", "fire_size_acres", "size_class"]].head()

## Spatial Analysis — Fires On and Near Tribal Lands

In [ ]:
def identify_fires_near_tribal_lands(
    fire_gdf: gpd.GeoDataFrame,
    tribal_gdf: gpd.GeoDataFrame,
    buffer_distance_km: float = 5,
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    """
    Identify fires that occurred on or within buffer_distance_km of tribal lands.

    Uses Albers Equal Area (EPSG:5070) for accurate distance buffering across CONUS.
    Returns GeoDataFrames in EPSG:4326.

    Parameters
    ----------
    fire_gdf           : MTBS fire perimeters
    tribal_gdf         : Tribal land boundaries
    buffer_distance_km : Distance in km to define "near"

    Returns
    -------
    fires_on_tribal   : fires intersecting tribal lands directly
    fires_near_tribal : fires within the buffer zone
    """
    print("Reprojecting to Albers Equal Area for accurate buffering...")
    fire_proj   = geo_utils.to_projected(geo_utils.ensure_crs(fire_gdf,   constants.CRS_GEOGRAPHIC))
    tribal_proj = geo_utils.to_projected(geo_utils.ensure_crs(tribal_gdf, constants.CRS_GEOGRAPHIC))

    # Fires directly ON tribal lands
    fires_on = gpd.sjoin(
        fire_proj,
        tribal_proj[["name", "namelsad", "geometry"]].rename(
            columns={"name": "NAME", "namelsad": "NAMELSAD"}
        ),
        how="inner",
        predicate="intersects",
    ).to_crs(constants.CRS_GEOGRAPHIC)

    # Buffered tribal lands for "near" analysis
    tribal_buffered = tribal_proj.copy()
    tribal_buffered["geometry"] = tribal_proj.buffer(buffer_distance_km * 1000)

    fires_near = gpd.sjoin(
        fire_proj,
        tribal_buffered[["name", "namelsad", "geometry"]].rename(
            columns={"name": "NAME", "namelsad": "NAMELSAD"}
        ),
        how="inner",
        predicate="within",
    ).to_crs(constants.CRS_GEOGRAPHIC)

    # Restore original case for downstream consistency
    for gdf in [fires_on, fires_near]:
        if "ig_date" in gdf.columns:
            gdf["Ig_Date"] = gdf["ig_date"]
        if "burnbndac" in gdf.columns:
            gdf["BurnBndAc"] = gdf["burnbndac"]

    print(f"Fires ON tribal lands      : {len(fires_on):,}")
    print(f"Fires within {buffer_distance_km} km of tribal lands: {len(fires_near):,}")
    return fires_on, fires_near

In [ ]:
fires_on_tribal, fires_near_tribal = identify_fires_near_tribal_lands(
    fire_gdf,
    tribal_lands,
    buffer_distance_km=5,
)

## Temporal Analysis

In [ ]:
# Annual counts and acres
fires_on_per_year   = fires_on_tribal.groupby("fire_year").size().rename("fire_count")
acres_on_per_year   = fires_on_tribal.groupby("fire_year")["BurnBndAc"].sum().rename("acres_burned")
fires_near_per_year = fires_near_tribal.groupby("fire_year").size().rename("fire_count")
acres_near_per_year = fires_near_tribal.groupby("fire_year")["BurnBndAc"].sum().rename("acres_burned")

In [ ]:
# Fires ON tribal lands (frequency + acres)
fig, ax1 = plt.subplots(figsize=(13, 5))

ax1.plot(
    fires_on_per_year.index, fires_on_per_year.values,
    marker="o", linewidth=2, color=styles.EMBER_RED, label="Number of Fires",
)
ax1.set_xlabel("Year")
ax1.set_ylabel("Number of Fires", color=styles.EMBER_RED)
ax1.tick_params(axis="y", labelcolor=styles.EMBER_RED)
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    acres_on_per_year.index, acres_on_per_year.values,
    marker="s", linestyle="--", linewidth=2, color=styles.SKY_BLUE, label="Acres Burned",
)
ax2.set_ylabel("Acres Burned", color=styles.SKY_BLUE)
ax2.tick_params(axis="y", labelcolor=styles.SKY_BLUE)

lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc="upper left")
plt.title("MTBS Fires ON Tribal Lands: Frequency and Acres Burned (1984–2024)")
plt.tight_layout()

charts.save_figure(fig, "outputs/figures/fires_on_tribal_timeline.png")
plt.show()

In [ ]:
# Fires ON Tribal lands vs Near
fig, ax1 = plt.subplots(figsize=(13, 5))

ax1.plot(fires_on_per_year.index,   fires_on_per_year.values,
         marker="o", color=styles.EMBER_RED,   label="Fires ON")
ax1.plot(fires_near_per_year.index, fires_near_per_year.values,
         marker="o", color=styles.FIRE_ORANGE, label="Fires NEAR")
ax1.set_xlabel("Year")
ax1.set_ylabel("Number of Fires")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(acres_on_per_year.index,   acres_on_per_year.values,
         linestyle="--", marker="s", color="darkred",    label="Acres ON")
ax2.plot(acres_near_per_year.index, acres_near_per_year.values,
         linestyle="--", marker="s", color="darkorange", label="Acres NEAR")
ax2.set_ylabel("Acres Burned")

lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc="upper left")
plt.title("MTBS Fires ON vs NEAR Tribal Lands")
plt.tight_layout()

charts.save_figure(fig, "outputs/figures/fires_on_vs_near_tribal.png")
plt.show()

## Statistics by Tribe

In [ ]:
def generate_fire_statistics(fire_gdf: gpd.GeoDataFrame, label: str = "MTBS Fires") -> None:
    """Print summary statistics for an MTBS fire GeoDataFrame."""
    df = fire_gdf.copy()
    total       = len(df)
    year_range  = (df["fire_year"].min(), df["fire_year"].max())
    total_acres = df["BurnBndAc"].sum()

    print(f"\n{'='*60}")
    print(f"FIRE STATISTICS: {label}")
    print(f"{'='*60}")
    print(f"Total fires          : {total:,}")
    print(f"Date range           : {year_range[0]} – {year_range[1]}")
    print(f"Total acres burned   : {total_acres:,.0f}")
    print(f"Average fire size    : {df['BurnBndAc'].mean():,.1f} acres")
    print(f"Median fire size     : {df['BurnBndAc'].median():,.1f} acres")
    print(f"Largest fire         : {df['BurnBndAc'].max():,.0f} acres")
    fires_per_year = df.groupby("fire_year").size()
    print(f"Avg fires/year       : {fires_per_year.mean():.1f}")

    if "NAME" in df.columns:
        top_tribes = df["NAME"].value_counts().head(10)
        print("\nTop 10 tribal lands by fire count:")
        for tribe, count in top_tribes.items():
            print(f"  {tribe}: {count:,} ({count/total*100:.1f}%)")


generate_fire_statistics(fires_on_tribal,   "Fires ON Tribal Lands")
generate_fire_statistics(fires_near_tribal, "Fires NEAR Tribal Lands (5 km buffer)")

In [ ]:
# Acres burned per Tribe
acres_per_tribe = (
    fires_on_tribal.groupby("NAME")["BurnBndAc"]
    .sum()
    .sort_values(ascending=False)
)
print("Top 10 tribal lands by total acres burned:")
print(acres_per_tribe.head(10).to_string())

In [ ]:
# 10 largest fires
top_fires = (
    fires_on_tribal
    .sort_values("BurnBndAc", ascending=False)
    .head(10)
    [["event_id", "incid_name", "BurnBndAc", "fire_year", "NAME"]]
)
print("Top 10 largest fires on tribal lands:")
print(top_fires.to_string(index=False))

In [ ]:
# Bar chart for Tribes by acres burned
top_n = 10
top_acres = acres_per_tribe.head(top_n)

fig, ax = plt.subplots(figsize=(12, 6))
top_acres.sort_values().plot(
    kind="barh", ax=ax,
    color=styles.FIRE_ORANGE, alpha=0.85,
)
ax.set_xlabel("Total Acres Burned (1984–2024)")
ax.set_title(f"Top {top_n} Tribal Lands by Total Acres Burned (MTBS)")
ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x:,.0f}")
)
sns.despine(ax=ax)
plt.tight_layout()

charts.save_figure(fig, "outputs/figures/top_tribes_acres_burned.png")
plt.show()

## Fire Risk by Tribal Land

In [ ]:
def analyze_fire_risk_by_tribal_land(
    fires_on: gpd.GeoDataFrame,
) -> pd.DataFrame:
    """
    Compute composite fire risk scores per tribal land unit.

    Risk score = fires_per_decade × avg_fire_size_acres
    This weights both frequency and magnitude.

    Returns
    -------
    DataFrame sorted by risk_score descending
    """
    if fires_on.empty:
        raise ValueError("No fires on tribal lands to analyze.")

    stats = (
        fires_on.groupby("NAME")
        .agg(
            fire_count=("event_id", "count"),
            total_acres=("BurnBndAc", "sum"),
            avg_fire_size=("BurnBndAc", "mean"),
            max_fire_size=("BurnBndAc", "max"),
            first_year=("fire_year", "min"),
            last_year=("fire_year", "max"),
        )
        .reset_index()
    )

    year_span = (stats["last_year"] - stats["first_year"] + 1).clip(lower=1)
    stats["avg_acres_per_year"]  = stats["total_acres"]  / year_span
    stats["fires_per_decade"]    = stats["fire_count"]   / (year_span / 10)
    stats["risk_score"]          = stats["fires_per_decade"] * stats["avg_fire_size"]

    return stats.sort_values("risk_score", ascending=False).reset_index(drop=True)


tribal_risk_stats = analyze_fire_risk_by_tribal_land(fires_on_tribal)

print("Top 20 Tribal Lands by Fire Risk Score")
print("=" * 80)
print(tribal_risk_stats.head(20).to_string(index=False))

## Maps

In [ ]:
# Static risk map
def plot_fire_risk_map(
    fires_on: gpd.GeoDataFrame,
    fires_near: gpd.GeoDataFrame,
    tribal: gpd.GeoDataFrame,
    risk_stats: pd.DataFrame,
    figsize: tuple = (13, 9),
) -> plt.Figure:
    """
    Static choropleth map: tribal lands shaded by risk score,
    overlaid with fire perimeter centroids.
    """
    tribal_3857    = tribal.to_crs(epsg=3857)
    fires_on_3857  = fires_on.to_crs(epsg=3857)
    fires_near_3857 = fires_near.to_crs(epsg=3857)

    tribal_risk = tribal_3857.merge(
        risk_stats[["NAME", "risk_score"]],
        on="NAME",
        how="left",
    )
    tribal_risk["risk_score"] = tribal_risk["risk_score"].fillna(0)

    fig, ax = plt.subplots(figsize=figsize)

    tribal_risk.plot(
        column="risk_score",
        cmap="OrRd",
        linewidth=0.6,
        edgecolor=styles.CHARCOAL,
        legend=True,
        legend_kwds={"label": "Fire Risk Score (frequency × avg size)"},
        ax=ax,
    )

    # Plot perimeter centroids for visual density
    if not fires_near_3857.empty:
        fires_near_3857.centroid.plot(
            ax=ax, color=styles.FIRE_ORANGE,
            markersize=8, alpha=0.45, label="Fires Near Tribal Lands",
        )
    if not fires_on_3857.empty:
        fires_on_3857.centroid.plot(
            ax=ax, color=styles.SKY_BLUE,
            markersize=8, alpha=0.55, label="Fires On Tribal Lands",
        )

    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
    ax.set_axis_off()
    ax.set_title(
        "Fire Risk on and Near Tribal Lands (MTBS 1984–2024)",
        fontsize=15, fontweight="bold",
    )
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, loc="lower left", fontsize=9)

    plt.tight_layout()
    return fig


fig = plot_fire_risk_map(
    fires_on_tribal, fires_near_tribal,
    tribal_lands, tribal_risk_stats,
)
charts.save_figure(fig, "outputs/figures/fire_risk_map.png")
plt.show()

## Multivariate Analysis

In [ ]:
def create_fire_pairplot(
    fires_on: gpd.GeoDataFrame,
    fires_near: gpd.GeoDataFrame,
    top_n: int = 10,
) -> None:
    """
    Seaborn pairplot comparing fire metrics for the top N tribal lands.
    Uses log-transformed axes to handle skewed fire size distributions.
    """
    def summarize(gdf, label):
        return (
            gdf.groupby("NAME")
            .agg(
                fire_count=("BurnBndAc", "count"),
                total_acres=("BurnBndAc", "sum"),
                avg_fire_size=("BurnBndAc", "mean"),
            )
            .reset_index()
            .assign(Type=label)
        )

    on_summary   = summarize(fires_on,   "On Tribal Lands")
    near_summary = summarize(fires_near, "Near Tribal Lands")
    combined = pd.concat([on_summary, near_summary], ignore_index=True)

    top_tribes = on_summary.nlargest(top_n, "fire_count")["NAME"]
    combined   = combined[combined["NAME"].isin(top_tribes)]

    combined["log_total_acres"]    = np.log1p(combined["total_acres"])
    combined["log_avg_fire_size"]  = np.log1p(combined["avg_fire_size"])

    g = sns.pairplot(
        combined[["fire_count", "log_total_acres", "log_avg_fire_size", "Type"]],
        hue="Type",
        palette={"On Tribal Lands": styles.EMBER_RED, "Near Tribal Lands": styles.FIRE_ORANGE},
        diag_kind="kde",
    )
    g.fig.suptitle(
        f"Fire Metrics for Top {top_n} Tribal Lands by Fire Count",
        fontsize=14, fontweight="bold", y=1.02,
    )
    charts.save_figure(g.fig, "outputs/figures/fire_pairplot.png")
    plt.show()


create_fire_pairplot(fires_on_tribal, fires_near_tribal, top_n=10)

## Exports

In [ ]:
# Export fire statistics to CSV 
export_cols = ["event_id", "incid_name", "ig_date", "fire_year", "BurnBndAc", "NAME"]
export_path = constants.OUTPUTS_DIR / "fires_on_tribal_stats.csv"

fires_on_tribal[[c for c in export_cols if c in fires_on_tribal.columns]].to_csv(
    export_path, index=False
)
print(f"Stats exported → {export_path.relative_to(REPO_ROOT)}")

# Export risk stats
risk_path = constants.OUTPUTS_DIR / "tribal_fire_risk_scores.csv"
tribal_risk_stats.to_csv(risk_path, index=False)
print(f"Risk scores exported {risk_path.relative_to(REPO_ROOT)}")

## Summary report

*(Fill in after running the notebook with your data.)*

Findings to address:
- Which tribal lands show the highest composite risk scores, and what drives that score (frequency vs. size)?
- Are there visible trend inflections — years where frequency or acres burned increased significantly?
- What limitations apply to this analysis given that Census-defined tribal boundaries may
  not reflect the full extent of tribal jurisdiction or land stewardship?

---
## References

In [ ]:
print(generate_citations(["mtbs", "census_aiannh"]))